In [137]:
import os
import time
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import LabelEncoder, StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import f1_score, accuracy_score

import pickle


In [138]:
def generate_object_features(n_features, n_samples, rng):
    """Генерирует признаки для одного объекта с 4 типами данных"""
    types = ["binary", "nominal", "ordinal", "quantitative"]
    data = {}
    for i in range(n_features):
        t = types[i % 4]
        col_name = f"feat_{i}_{t}"
        
        if t == "binary":
            data[col_name] = rng.integers(0, 2, size=n_samples)
        elif t == "nominal":
            data[col_name] = rng.choice([1, 2, 3, 4], size=n_samples)
        elif t == "ordinal":
            data[col_name] = rng.integers(0, 5, size=n_samples)
        elif t == "quantitative":
            data[col_name] = rng.normal(loc=0.0, scale=1.0, size=n_samples)
            
    return pd.DataFrame(data)

In [139]:
def custom_collision_rule(obj1: pd.DataFrame, obj2: pd.DataFrame) -> pd.Series:
    """
    Вычисляет факт коллизии между двумя объектами на основе их признаков.
    
    Логика правила:
    1. Количественные признаки: если евклидово расстояние по числовым признакам < порога → риск коллизии
    2. Номинальные признаки: если есть совпадение хотя бы одного номинального признака → повышенный риск
    3. Порядковые признаки: если разница по порядковым признакам ≤ 1 → объекты "близки" по состоянию
    4. Бинарные признаки: если оба объекта активны (значение 1) → возможна коллизия
    
    Коллизия = 1, если выполняются ≥ 2 из 4 условий выше.
    """
    n_samples = len(obj1)
    collision_conditions = np.zeros(n_samples, dtype=int)
    
    quant_cols1 = [c for c in obj1.columns if c.endswith('_quantitative')]
    quant_cols2 = [c for c in obj2.columns if c.endswith('_quantitative')]
    
    if quant_cols1 and quant_cols2:
        dist = np.zeros(n_samples)
        for c1, c2 in zip(quant_cols1, quant_cols2):
            std1 = obj1[c1].std() if obj1[c1].std() > 0 else 1
            std2 = obj2[c2].std() if obj2[c2].std() > 0 else 1
            diff = ((obj1[c1] / std1) - (obj2[c2] / std2)) ** 2
            dist += diff
        dist = np.sqrt(dist / len(quant_cols1))
        collision_conditions += (dist < 1.5).astype(int)
    
    nominal_cols1 = [c for c in obj1.columns if c.endswith('_nominal')]
    nominal_cols2 = [c for c in obj2.columns if c.endswith('_nominal')]
    
    if nominal_cols1 and nominal_cols2:
        nominal_match = np.zeros(n_samples, dtype=bool)
        for c1, c2 in zip(nominal_cols1, nominal_cols2):
            nominal_match |= (obj1[c1] == obj2[c2])
        collision_conditions += nominal_match.astype(int)
    
    ordinal_cols1 = [c for c in obj1.columns if c.endswith('_ordinal')]
    ordinal_cols2 = [c for c in obj2.columns if c.endswith('_ordinal')]
    
    if ordinal_cols1 and ordinal_cols2:
        ordinal_close = np.zeros(n_samples, dtype=bool)
        for c1, c2 in zip(ordinal_cols1, ordinal_cols2):
            ordinal_close |= (np.abs(obj1[c1] - obj2[c2]) <= 1)
        collision_conditions += ordinal_close.astype(int)
    
    binary_cols1 = [c for c in obj1.columns if c.endswith('_binary')]
    binary_cols2 = [c for c in obj2.columns if c.endswith('_binary')]
    
    if binary_cols1 and binary_cols2:
        both_active = np.zeros(n_samples, dtype=bool)
        for c1, c2 in zip(binary_cols1, binary_cols2):
            both_active |= ((obj1[c1] == 1) & (obj2[c2] == 1))
        collision_conditions += both_active.astype(int)
    
    return pd.Series(collision_conditions >= 2, index=obj1.index)

In [140]:
sizes = [50, 300, 750, 1500]
n_features_list = [5, 9, 12]

In [141]:
for size in sizes:
    for nf in n_features_list:
        rng = np.random.default_rng(seed=67)
        
        n_obj1 = nf // 2 + (nf % 2)
        n_obj2 = nf // 2
        
        df1 = generate_object_features(n_obj1, size, rng)
        df2 = generate_object_features(n_obj2, size, rng)
        
        df1.columns = [f"obj1_{c}" for c in df1.columns]
        df2.columns = [f"obj2_{c}" for c in df2.columns]
        
        df = pd.concat([df1, df2], axis=1)
        df["collision"] = custom_collision_rule(df[df1.columns], df[df2.columns])
        
        fname = f"datasets/ds_size{size}_feat{nf}.csv"
        df.to_csv(fname, index=False)
        print(f"{fname} | Shape: {df.shape} | Collision rate: {df['collision'].mean():.2%}")

datasets/ds_size50_feat5.csv | Shape: (50, 6) | Collision rate: 12.00%
datasets/ds_size50_feat9.csv | Shape: (50, 10) | Collision rate: 68.00%
datasets/ds_size50_feat12.csv | Shape: (50, 13) | Collision rate: 82.00%
datasets/ds_size300_feat5.csv | Shape: (300, 6) | Collision rate: 4.00%
datasets/ds_size300_feat9.csv | Shape: (300, 10) | Collision rate: 63.33%
datasets/ds_size300_feat12.csv | Shape: (300, 13) | Collision rate: 74.00%
datasets/ds_size750_feat5.csv | Shape: (750, 6) | Collision rate: 6.27%
datasets/ds_size750_feat9.csv | Shape: (750, 10) | Collision rate: 58.53%
datasets/ds_size750_feat12.csv | Shape: (750, 13) | Collision rate: 72.40%
datasets/ds_size1500_feat5.csv | Shape: (1500, 6) | Collision rate: 6.00%
datasets/ds_size1500_feat9.csv | Shape: (1500, 10) | Collision rate: 57.80%
datasets/ds_size1500_feat12.csv | Shape: (1500, 13) | Collision rate: 71.60%


In [142]:
def get_feature_groups(columns):
    binary = [c for c in columns if c.endswith('_binary')]
    nominal = [c for c in columns if c.endswith('_nominal')]
    ordinal = [c for c in columns if c.endswith('_ordinal')]
    quantitative = [c for c in columns if c.endswith('_quantitative')]
    return binary, nominal, ordinal, quantitative


def make_pipeline(model, feature_columns):
    binary, nominal, ordinal, quantitative = get_feature_groups(feature_columns)
    transformers = []
    if quantitative:
        transformers.append(('num', StandardScaler(), quantitative))
    if nominal:
        transformers.append(('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), nominal))
    preprocessor = ColumnTransformer(transformers=transformers, remainder='passthrough')
    return Pipeline([('preprocessor', preprocessor), ('classifier', model)])


def evaluate(pipe, X, y):
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)
    t0 = time.time()
    pipe.fit(X_train, y_train)
    fit_time = time.time() - t0
    t1 = time.time()
    y_pred = pipe.predict(X_test)
    pred_time = time.time() - t1
    return {
        'f1': f1_score(y_test, y_pred),
        'accuracy': accuracy_score(y_test, y_pred),
        'fit_time': fit_time,
        'predict_time': pred_time
    }

Обоснование выбора 4 классических методов:
1. LogisticRegression: быстрая линейная модель, интерпретируема, служит надежным бенчмарком.
2. DecisionTreeClassifier: не требует масштабирования, устойчив к нелинейным зависимостям.
3. KNeighborsClassifier: интуитивно подходит для задачи коллизий, так как близкие состояния в пространстве признаков часто ведут к схожим исходам.
4. RandomForestClassifier: ансамблевый метод, устойчив к переобучению, стабильно работает на выборках разного объема.

Выбранная метрика: F1-score, так как классы могут быть несбалансированы (частота коллизий от 4% до 82%), и нам важен баланс между precision и recall.

In [143]:
models = {
    'logreg': LogisticRegression(max_iter=1000, random_state=42),
    'dtree': DecisionTreeClassifier(random_state=42),
    'knn': KNeighborsClassifier(),
    'rf': RandomForestClassifier(n_estimators=50, random_state=42)
}

In [144]:
all_results = []
for size in sizes:
    for nf in n_features_list:
        path = f"datasets/ds_size{size}_feat{nf}.csv"
        df = pd.read_csv(path)
        X = df.drop('collision', axis=1)
        y = df['collision']
        cols = X.columns.tolist()
        
        for name, model in models.items():
            pipe = make_pipeline(model, cols)
            metrics = evaluate(pipe, X, y)
            all_results.append({'size': size, 'n_features': nf, 'model': name, **metrics})

results_df = pd.DataFrame(all_results)
results_sorted = results_df.sort_values(by=['model', 'size', 'n_features'])

results_sorted

C:\Users\danii\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 due to no true nor predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\danii\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 due to no true nor predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


,size,n_features,model,f1,accuracy,fit_time,predict_time
1,50,5,dtree,0.000000,1.000000,0.005558,0.003444
5,50,9,dtree,0.909091,0.866667,0.006557,0.003394
9,50,12,dtree,0.833333,0.733333,0.007473,0.004291
13,300,5,dtree,0.500000,0.977778,0.004159,0.002455
17,300,9,dtree,0.765217,0.700000,0.006878,0.003453
21,300,12,dtree,0.748092,0.633333,0.007862,0.004062
25,750,5,dtree,1.000000,1.000000,0.004461,0.002072
29,750,9,dtree,0.769231,0.720000,0.008776,0.003520
33,750,12,dtree,0.809816,0.724444,0.009446,0.003904
37,1500,5,dtree,1.000000,1.000000,0.004671,0.002487


In [145]:
small_df = results_df[results_df['size'] <= 100]
summary_small = small_df.groupby('model').agg({'f1': 'mean', 'accuracy': 'mean', 'fit_time': 'mean', 'predict_time': 'mean'}).reset_index()
display(summary_small.round(4))
top3 = summary_small.nsmallest(3, 'fit_time')['model'].tolist()

summary = results_df.groupby('model').agg({'f1': 'mean', 'accuracy': 'mean', 'fit_time': 'mean', 'predict_time': 'mean'}).reset_index()
display(summary.round(4))

,model,f1,accuracy,fit_time,predict_time
0,dtree,0.5808,0.8667,0.0065,0.0037
1,knn,0.5572,0.8000,0.0070,0.0056
2,logreg,0.5396,0.8000,0.0118,0.0039
3,rf,0.5741,0.8222,0.0712,0.0080


,model,f1,accuracy,fit_time,predict_time
0,dtree,0.7355,0.8089,0.0073,0.0034
1,knn,0.6755,0.8215,0.0065,0.0073
2,logreg,0.5195,0.7680,0.0125,0.0036
3,rf,0.7966,0.8383,0.0789,0.0086


In [146]:
param_grids = {
    'knn': {'classifier__n_neighbors': [3, 5, 7]},
    'dtree': {
        'classifier__max_depth': [None, 3, 5, 10, 15, 20],
        'classifier__min_samples_split': [2, 5, 10, 15, 20]
    },
    'rf': {
        'classifier__n_estimators': [50, 100],
        'classifier__max_depth': [1, 3, 5, None]
    },
}

top3 = {
    'knn': {'classifier__n_neighbors': [3, 5, 7]},
    'dtree': DecisionTreeClassifier(random_state=42),
    'rf': RandomForestClassifier(n_estimators=50, random_state=42)
}

In [147]:
tuning_results = []
sizes = {50: 5, 1500: 12}
for model_name in top3:
    for size, nf in sizes.items():
        path = f"datasets/ds_size{size}_feat{nf}.csv"
        if not os.path.exists(path):
            continue
        df = pd.read_csv(path)
        X = df.drop('collision', axis=1)
        y = df['collision']
        cols = X.columns.tolist()
        base_model = models[model_name]
        pipe = make_pipeline(base_model, cols)
        grid = GridSearchCV(pipe, param_grids[model_name], cv=3, scoring='f1', n_jobs=-1)
        t0 = time.time()
        grid.fit(X, y)
        elapsed = time.time() - t0
        
        tuning_results.append({'model': model_name, 'size': size, 'n_features': nf, 'best_params': grid.best_params_, 'best_f1': grid.best_score_, 'tuning_time': elapsed})
        print(f"Tuning {model_name} | Size:{size} | Time:{elapsed:.2f}s | F1:{grid.best_score_:.3f}")

tuning_df = pd.DataFrame(tuning_results)
tuning_df

Tuning knn | Size:50 | Time:0.05s | F1:0.000
Tuning knn | Size:1500 | Time:0.13s | F1:0.856
Tuning dtree | Size:50 | Time:0.18s | F1:0.500
Tuning dtree | Size:1500 | Time:0.28s | F1:0.834
Tuning rf | Size:50 | Time:0.45s | F1:0.444
Tuning rf | Size:1500 | Time:0.73s | F1:0.860


,model,size,n_features,best_params,best_f1,tuning_time
0,knn,50,5,{'classifier__n_neighbors': 3},0.000000,0.046239
1,knn,1500,12,{'classifier__n_neighbors': 7},0.856283,0.129588
2,dtree,50,5,"{'classifier__max_depth': None, 'classifier__m...",0.500000,0.184462
3,dtree,1500,12,"{'classifier__max_depth': 5, 'classifier__min_...",0.834313,0.282204
4,rf,50,5,"{'classifier__max_depth': 5, 'classifier__n_es...",0.444444,0.445508
5,rf,1500,12,"{'classifier__max_depth': None, 'classifier__n...",0.859506,0.730008


knn на мылах данных просто не работает с gridsearch

In [148]:
summary_table = tuning_df.groupby('model').agg({
    'best_f1': ['mean', 'std', 'max'],
    'tuning_time': ['mean', 'sum', 'max']
}).round(4)

summary_table

best_f1                 tuning_time                
         mean     std     max        mean     sum     max
model                                                    
dtree  0.6672  0.2364  0.8343      0.2333  0.4667  0.2822
knn    0.4281  0.6055  0.8563      0.0879  0.1758  0.1296
rf     0.6520  0.2935  0.8595      0.5878  1.1755  0.7300

In [149]:
print("ИТОГОВЫЕ РЕЗУЛЬТАТЫ GRIDSEARCHCV")

for model in tuning_df['model'].unique():
    model_data = tuning_df[tuning_df['model'] == model]
    print(f"\n{model.upper()}:")
    print(f"   Средний F1: {model_data['best_f1'].mean():.4f} ± {model_data['best_f1'].std():.4f}")
    print(f"   Лучший F1:  {model_data['best_f1'].max():.4f}")
    print(f"   Общее время: {model_data['tuning_time'].sum():.3f}s")
    print(f"   Лучший параметр: {model_data.loc[model_data['best_f1'].idxmax(), 'best_params']}")

ИТОГОВЫЕ РЕЗУЛЬТАТЫ GRIDSEARCHCV

KNN:
   Средний F1: 0.4281 ± 0.6055
   Лучший F1:  0.8563
   Общее время: 0.176s
   Лучший параметр: {'classifier__n_neighbors': 7}

DTREE:
   Средний F1: 0.6672 ± 0.2364
   Лучший F1:  0.8343
   Общее время: 0.467s
   Лучший параметр: {'classifier__max_depth': 5, 'classifier__min_samples_split': 2}

RF:
   Средний F1: 0.6520 ± 0.2935
   Лучший F1:  0.8595
   Общее время: 1.176s
   Лучший параметр: {'classifier__max_depth': None, 'classifier__n_estimators': 100}


`DessisionTree` - лучший выбор, затем идет `RF` (хотя `RF` выдает лучший `F1`, при этом он показывает наибольшее время работы, а согласно заданию надо выбрать алгоритм с минимальными ресурсами, в том числе временными)

In [150]:
best_params = {
    'small': {
        'dtree': {'classifier__max_depth': 5},
        'knn': {'classifier__n_neighbors': 3}
    },
    'large': {
        'dtree': {'classifier__max_depth': 10},
        'knn': {'classifier__n_neighbors': 7}
    }
}

os.makedirs('models', exist_ok=True)

data_paths = {
    'small': 'datasets/ds_size50_feat5.csv',
    'large': 'datasets/ds_size1500_feat12.csv'
}

for size_name, path in data_paths.items():
    df = pd.read_csv(path)
    X = df.drop('collision', axis=1)
    y = df['collision']
    cols = X.columns.tolist()

    # Берем параметры для текущего размера датасета
    size_params = best_params[size_name]
    
    for model_name, params in size_params.items():
        pipe = make_pipeline(models[model_name], cols)
        pipe.set_params(**params)
        pipe.fit(X, y)

        fname = f"models/{model_name}_{size_name}.pkl"
        joblib.dump(pipe, fname)
        print(f"✓ Saved: {fname} with params {params}")

✓ Saved: models/dtree_small.pkl with params {'classifier__max_depth': 5}
✓ Saved: models/knn_small.pkl with params {'classifier__n_neighbors': 3}
✓ Saved: models/dtree_large.pkl with params {'classifier__max_depth': 10}
✓ Saved: models/knn_large.pkl with params {'classifier__n_neighbors': 7}
